In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time

def get_vacancy_ids(text='python', pages=20, area=1, experience=None):
    base_url = 'https://api.hh.ru/vacancies'
    ids = []
    for page in range(pages):
        params = {
            'text': text,
            'area': area,
            'per_page': 100,
            'page': page
        }
        if experience:
            params['experience'] = experience

        res = requests.get(base_url, params=params)
        if res.status_code != 200:
            print(f"❌ Failed on page {page} [experience={experience}]")
            continue
        data = res.json()
        page_ids = [item['id'] for item in data.get('items', [])]
        if not page_ids:
            break
        ids += page_ids
        print(f"✅ Page {page} OK, {len(page_ids)} IDs fetched (experience={experience})")
        time.sleep(0.3)
    return ids

def parse_salary(salary):
    if not salary:
        return "Not specified"
    _from = salary.get('from')
    _to = salary.get('to')
    currency = salary.get('currency', '')
    if _from and _to:
        return f"{_from}–{_to} {currency}"
    elif _from:
        return f"From {_from} {currency}"
    elif _to:
        return f"Up to {_to} {currency}"
    return "Not specified"

def fetch_vacancy_details(vac_id):
    url = f"https://api.hh.ru/vacancies/{vac_id}"
    res = requests.get(url)
    if res.status_code != 200:
        return None
    data = res.json()
    return {
        'title': data.get('name'),
        'company': data.get('employer', {}).get('name'),
        'salary': parse_salary(data.get('salary')),
        'url': data.get('alternate_url'),
    }

def clean_text(text, prefix=None):
    if not text:
        return None
    text = text.strip()
    if prefix and text.startswith(prefix):
        return text.replace(prefix, '').strip()
    return text

def scrape_page_info(url):
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
        }
        res = requests.get(url, headers=headers)
        if res.status_code != 200:
            return {}

        soup = BeautifulSoup(res.content, 'html.parser')

        def get_text_by_qa(tag, data_qa):
            el = soup.find(tag, attrs={'data-qa': data_qa})
            return el.get_text(strip=True) if el else None

        exp = clean_text(get_text_by_qa('p', 'work-experience-text'), 'Опыт работы:')
        emp = clean_text(get_text_by_qa('div', 'common-employment-text'))
        sch = clean_text(get_text_by_qa('p', 'work-schedule-by-days-text'), 'График:')
        hrs = clean_text(get_text_by_qa('div', 'working-hours-text'), 'Рабочие часы:')

        return {
            'experience': normalize_experience(exp),
            'employment': normalize_employment(emp),
            'schedule': normalize_schedule(sch),
            'work_hours': normalize_hours(hrs)
        }

    except Exception as e:
        print(f"❌ Failed to scrape {url}: {e}")
        return {}

# 🧽 Normalization helpers

def normalize_experience(value):
    if not value:
        return None
    val = value.lower()

    if "без опыта" in val or "нет опыта" in val:
        return 0
    elif "1–3" in val or "1-3" in val:
        return 1
    elif "3–6" in val or "3-6" in val:
        return 3
    elif "более" in val or "6+" in val or "более 6" in val:
        return 6

    digits = ''.join(c for c in val if c.isdigit())
    return int(digits) if digits else None

def normalize_employment(value):
    if not value:
        return None
    val = value.lower()
    if "полная" in val:
        return "Полная"
    elif "частич" in val:
        return "Частичная"
    elif "проект" in val:
        return "Проектная"
    elif "стажировка" in val:
        return "Стажировка"
    return value

def normalize_schedule(value):
    if not value:
        return None
    val = value.lower()
    if "удалён" in val:
        return "Удалённый"
    elif "сменный" in val:
        return "Сменный"
    elif "гибкий" in val or "свободный" in val:
        return "Свободный"
    elif "5/2" in val:
        return "5/2"
    return value

def normalize_hours(value):
    if not value:
        return None
    digits = ''.join(c for c in value if c.isdigit())
    return digits if digits else value

def main():
    query = "python"
    print("🔍 Getting IDs for noExperience...")
    ids_1 = get_vacancy_ids(text=query, pages=20, area=1, experience="noExperience")

    print("🔍 Getting IDs for between1And3...")
    ids_2 = get_vacancy_ids(text=query, pages=20, area=1, experience="between1And3")

    print("🔍 Getting IDs for between3And6...")
    ids_3 = get_vacancy_ids(text=query, pages=20, area=1, experience="between3And6")

    print("🔍 Getting IDs for moreThan6...")
    ids_4 = get_vacancy_ids(text=query, pages=20, area=1, experience="moreThan6")

    all_ids = list(set(ids_1 + ids_2 + ids_3 + ids_4))
    print(f"🧠 Total unique vacancy IDs: {len(all_ids)}")

    results = []
    for i, vac_id in enumerate(all_ids):
        data = fetch_vacancy_details(vac_id)
        if data:
            extra = scrape_page_info(data['url'])
            data.update(extra)
            results.append(data)
        print(f"Fetched {i+1}/{len(all_ids)}", end='\r')
        time.sleep(0.2)

    df = pd.DataFrame(results)
    df.to_csv('python_vacancies.csv', index=False)
    print("\n✅ Saved to python_vacancies.csv")

if __name__ == '__main__':
    main()


🔍 Getting IDs for noExperience...
✅ Page 0 OK, 100 IDs fetched (experience=noExperience)
✅ Page 1 OK, 100 IDs fetched (experience=noExperience)
✅ Page 2 OK, 42 IDs fetched (experience=noExperience)
🔍 Getting IDs for between1And3...
✅ Page 0 OK, 100 IDs fetched (experience=between1And3)
✅ Page 1 OK, 100 IDs fetched (experience=between1And3)
✅ Page 2 OK, 100 IDs fetched (experience=between1And3)
✅ Page 3 OK, 100 IDs fetched (experience=between1And3)
✅ Page 4 OK, 100 IDs fetched (experience=between1And3)
✅ Page 5 OK, 100 IDs fetched (experience=between1And3)
✅ Page 6 OK, 100 IDs fetched (experience=between1And3)
✅ Page 7 OK, 100 IDs fetched (experience=between1And3)
✅ Page 8 OK, 100 IDs fetched (experience=between1And3)
✅ Page 9 OK, 100 IDs fetched (experience=between1And3)
✅ Page 10 OK, 100 IDs fetched (experience=between1And3)
✅ Page 11 OK, 100 IDs fetched (experience=between1And3)
✅ Page 12 OK, 100 IDs fetched (experience=between1And3)
✅ Page 13 OK, 100 IDs fetched (experience=between1